<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

creating pivot view sorted by region for the sampled_par.xlsx files, then plot by 
regions: st johns, avalon, western, central

In [1]:
import os
import openpyxl
import pandas as pd
import matplotlib.pyplot as plt
import math
import datetime
import matplotlib.patches as mpatches
import re
import numpy as np

In [2]:
increment = 19

In [3]:
# Parameters
increment = 24


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
folder = r"C:\Users\CAMG038492\OneDrive - WSP O365\Documents\Climate Data\NF Power GIS\Output Samples\uploaded"
paralist = [t[8:-5] for t in os.listdir(folder) if t.startswith("sampled_") and not t.endswith(".csv")]
# todo later:    storms tropicalstorms waves
para = paralist[increment:][0]
print(para)
path = os.path.join(folder, "sampled_"+para+".xlsx")
path

IndexError: list index out of range

In [ ]:
sheets = pd.read_excel(path, sheet_name=None)
#sheets = pd.read_excel(r"C:\Users\CAMG038492\OneDrive - WSP O365\Documents\Climate Data\NF Power GIS\Output Samples\uploaded\archive\sampled_sfcWindmax.xlsx", sheet_name = None)

In [ ]:
cdf = pd.DataFrame()
for name, df in sheets.items():
    if name != "metadata":
        for col in df.columns:
            if col not in cdf.columns:
                cdf[col] = df[col]
cdf.head()

In [ ]:
# region filtering
regions = {
    "St Johns" : ["St. John's", "St.John's"],
    "Avalon" : ['Avalon', 'Eastern', 'Burin'],
    "Central" : ['Central', 'Bonavista', 'Gander', 'Grand Falls'],
    "Western" : ['Western', 'Corner Brook', 'Stephenville']
}
mapping = {}
for v, synonyms in regions.items():
    for synonym in synonyms:
        mapping[synonym] = v  # map each synonym to the standard name

In [ ]:
cdf['Region_summarized'] = cdf['Region'].map(mapping)
cdf

In [ ]:
canrcm4list = ['humidex', 'pr-50mm','prsn-50mm', 'pr','prsn', 'ros', 'sfcWindmax','sfcWindmax-anwindgust63', 'ros-anrainonsnow1', 'sfcWindmax-anwindspeed63']
awkcanrcm4list = ['drydays','humidex30','pr50','prsn50','prfr']

In [ ]:
datacols = [t for t in cdf.columns if t.startswith(para) and not "45_" in t and not "126_" in t]
datacols


In [ ]:
if para in ['storms','tropicalstorms','flood','stormsurge','waves']:
    datacols = cdf.columns[7:-1]
    datacols = [t for t in datacols if '45_' not in t]
if para == 'waves':
    datacols = [t for t in datacols if 'corrected' in t and 'Hsavg' in t]
if para in canrcm4list:
    #if para not in ['prsn','pr']:
        #pass
    datacols = [s for s in datacols if 'annsum' not in s and 'annstd' not in s]
    #else:
    #datacols = [s for s in datacols if 'annsum' in s]
if para == 'sfcWindmax':
    datacols = [s for s in datacols if "anwindgust63" in s]
    datacols = [t for t in datacols if 'annp' not in t]
    para = 'sfcWindmax-anwindgust63'
if para == 'ros':
    datacols = [s for s in datacols if "anrainonsnow1_" in s and "annp" not in s]
    para = 'ros-anrainonsnow1'
if para == 'prsn-50mm':
    datacols = [s for s in datacols if "annp" not in s]


In [ ]:
datacols

In [ ]:
grouped_mean = cdf.groupby('Region_summarized')[datacols].mean()
grouped_min = cdf.groupby('Region_summarized')[datacols].min()
grouped_max = cdf.groupby('Region_summarized')[datacols].max()
grouped_mean.transpose().to_csv(os.path.join(folder,"plots",f"grouped_{para}_regionmean.csv"))
grouped_min.transpose().to_csv(os.path.join(folder,"plots",f"grouped_{para}_regionmin.csv"))
grouped_max.transpose().to_csv(os.path.join(folder,"plots",f"grouped_{para}_regionmax.csv"))

In [ ]:
# Dark mode style
plt.style.use('dark_background')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',
    'figure.facecolor': '#222222',
    'axes.facecolor': '#222222',
    'grid.color': '#555555'
})

In [ ]:
df_mean = grouped_mean
df_min = grouped_min
df_max = grouped_max
lowerr = df_mean-df_min
higerr = df_max-df_mean
regions = grouped_mean.index.tolist()
numeric_df = grouped_mean
df_mean

In [ ]:
order_map = {'min': 0, 'p5': 0, 'p10': 1, 'p50': 2, 'avg':3, 'p90': 4, 'p95': 5, 'max':6,'': 7}
labels_data = []
seen = []
for col in numeric_df.columns:
    if para == 'flashrate':
        scen = 'CESM2' if 'CESM2' in col else ('UKESM' if 'UKESM' in col else ('GISS' if 'GISS' in col else ''))
    elif para in canrcm4list:
        scen = 'min' if 'annmin' in col else ('max' if 'annmax' in col else ('p10' if 'p10' in col else ('p90' if 'p90' in col else('p50' if 'annmean' in col else ''))))
    elif 'p95' in col: scen = 'p95'
    elif 'p90' in col: scen = 'p90'
    elif 'p50' in col: scen = 'p50'
    elif 'p10' in col: scen = 'p10'
    elif 'p5_' in col: scen = 'p5'
    elif para == 'waves':
        scen = 'max' if 'max' in col else ('min' if 'min' in col else ('mean' if 'mean' in col else ''))
    else:
        if para in awkcanrcm4list:
            if "annmax" in col: scen = "max"
            elif "annmin" in col: scen = "min"
            else: scen = 'mean'
        else: scen = ''
    try: date = re.search(r'\d{4}-\d{4}', col).group(0)
    except: date = re.search(r'\d{6}-\d{6}', col).group(0)
    label = f'{scen}_{date}' if scen else date
    print(col,label)
    if label in seen:
        raise ValueError(f"duplicate found {label} for {col}, rerun")
    else:
        seen.append(label)
    labels_data.append({'col': col, 'scenario': scen, 'date': date, 'label': label, 'order': order_map.get(scen, 3)})

labels_data_sorted = sorted(labels_data, key=lambda x: (x['date'], x['order']))
sorted_cols = [d['col'] for d in labels_data_sorted]
sorted_labels = [d['label'] for d in labels_data_sorted]


In [ ]:
c_default = '#CCCCCC'
c_p5 = '#E69F00'
c_p50 = '#56B4E9'
c_p95 = '#009E73'
sorted_colors = [c_p95 if d['scenario'] == 'p95' or d['scenario'] == 'p90' or d['scenario'] == 'max'
                 else c_p50 if d['scenario'] == 'p50' or d['scenario'] == 'mean'
                 else c_p5 if d['scenario'] == 'p5' or d['scenario'] == 'p10' or d['scenario'] == 'min'
                 else c_default for d in labels_data_sorted]

In [ ]:
n = len(df_mean)
rows = int(math.ceil(math.sqrt(n)))
cols = int(math.ceil(n / rows))
fig, axes = plt.subplots(rows, cols, figsize=(cols*9, rows*7))
axes = axes.flatten() if n > 1 else [axes]

for i in range(n):
    vals = df_mean.iloc[i][sorted_cols].values

    whiskers = True
    if whiskers:
        low = lowerr.iloc[i][sorted_cols].values
        hig = higerr.iloc[i][sorted_cols].values
        xerr = [low, hig]
        bars = axes[i].barh(sorted_labels, vals, color=sorted_colors, xerr=xerr, capsize = 5, error_kw={'ecolor': 'white', 'elinewidth': 1})
    if not whiskers:
        bars = axes[i].barh(sorted_labels, vals, color=sorted_colors)

    axes[i].invert_yaxis()  # first label at top
    axes[i].set_title(regions[i])
    axes[i].set_xlabel('Value')
    axes[i].grid(axis='x', linestyle='--', alpha=0.5)

    if whiskers:
        try:
            max_end = np.nanmax(vals + hig)
            texts = [f'{v:.3f}' for v in vals]
            margin = (max(len(s) for s in texts))
            axes[i].set_xlim(0, max_end + margin)
        except: pass

    for j,bar in enumerate(bars):
        width = bar.get_width()
        y = bar.get_y() + bar.get_height() / 2.
        offset = bar.get_height()*0.3

        if whiskers:
            hwid = hig[j] + 1.5
            axes[i].text(width+hwid, y-offset, f'min {width-low[j]:.3f}', va='center', ha='left', fontsize=9, fontweight='bold', color='white')
            axes[i].text(width+hwid, y, f'mean {width:.3f}', va='center', ha='left', fontsize=9, fontweight='bold', color='white')
            axes[i].text(width+hwid, y+offset, f'max {width+hig[j]:.3f}', va='center', ha='left', fontsize=9, fontweight='bold', color='white')

        if not whiskers:
            axes[i].text(width, y, f'{width:.3f}', va='center', ha='left', fontsize=9, fontweight='bold', color='white')


for j in range(n, len(axes)):
    fig.delaxes(axes[j])

legend_handles = [
    mpatches.Patch(color=c_p5, label='p5'),
    mpatches.Patch(color=c_p50, label='p50'),
    mpatches.Patch(color=c_p95, label='p95'),
    mpatches.Patch(color=c_default, label='other')
]
leg = fig.legend(handles=legend_handles, loc='upper right', title='Scenario', fontsize=12, title_fontsize=14)
for text in leg.get_texts():
    text.set_color('white')
leg.get_title().set_color('white')

plt.tight_layout(rect=[0, 0, .95, .95])
plt.suptitle(para + "  " + str(datetime.datetime.now()))
plt.savefig(os.path.join(folder,"plots",f"plotted_whisker_{para}.png"))
plt.show()


In [ ]:
increment += 1
print(f"next {paralist[increment:][0]}")